# MAI391 — Red Wine Quality: Model Implementation & Baseline
**Author:** Khuê (Member 4 - Model Implementation & Baseline)
**Course:** MAI391 — Mathematics for Machine Learning
**Input:** `data/train.csv`, `data/test.csv` (sinh từ Khoa - M1)
**Output:** Model `models/baseline_linear_regression.joblib`, kết quả metrics & predictions trong `reports/`

---

## 1. Import thư viện và xác định Project Root
Tự động tìm đường dẫn thư mục gốc repo để chạy tương thích trên mọi môi trường (VS Code / Jupyter Notebook).

In [1]:
import json
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for parent in (start, *start.parents):
        if (parent / "data").is_dir() and (parent / "notebooks").is_dir():
            return parent
    if start.name == "notebooks":
        return start.parent
    if (start / "notebooks").is_dir():
        return start
    raise FileNotFoundError("Không tìm thấy thư mục gốc repo chứa data/ và notebooks/")

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
MODELS_DIR = ROOT / "models"

print(f"Project root: {ROOT}")

Project root: E:\-MAI391-Wine-Quality-Prediction


## 2. Load dữ liệu Train & Test chuẩn từ `data/`
Tuân thủ quy ước nhóm: Không split lại dữ liệu, đọc trực tiếp `data/train.csv` và `data/test.csv`.

In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

assert train_path.exists(), f"Thiếu file {train_path}. Hãy chạy notebook Khoa_Dataset_Preprocessing.ipynb trước!"
assert test_path.exists(), f"Thiếu file {test_path}."

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Đọc thứ tự 11 features từ feature_order.json do M1/M3 thống nhất
feature_order_path = REPORTS_DIR / "feature_order.json"
if feature_order_path.exists():
    with open(feature_order_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
        feature_names = meta["features"]
        target_col = meta["target"]
else:
    target_col = "quality"
    feature_names = [c for c in train_df.columns if c != target_col]

X_train = train_df[feature_names]
y_train = train_df[target_col]
X_test = test_df[feature_names]
y_test = test_df[target_col]

print(f"Train samples: {len(X_train)} | Features: {X_train.shape[1]}")
print(f"Test samples:  {len(X_test)} | Features: {X_test.shape[1]}")
print("Feature names in order:", feature_names)

Train samples: 1279 | Features: 11
Test samples:  320 | Features: 11
Feature names in order: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


## 3. Huấn luyện mô hình Baseline (Multiple Linear Regression) và đo thời gian
Sử dụng `sklearn.linear_model.LinearRegression` với thuật toán Least Squares tối ưu hàm mất mát Mean Squared Error (MSE).

In [3]:
model = LinearRegression()

# Đo thời gian huấn luyện chính xác bằng time.perf_counter()
t_train_start = time.perf_counter()
model.fit(X_train, y_train)
t_train_end = time.perf_counter()
train_time_ms = (t_train_end - t_train_start) * 1000

print(f"Huấn luyện thành công trong {train_time_ms:.3f} ms")

Huấn luyện thành công trong 64.069 ms


## 4. Dự đoán và đo thời gian suy luận (Prediction Time)

In [4]:
t_pred_start = time.perf_counter()
y_test_pred = model.predict(X_test)
t_pred_end = time.perf_counter()
pred_time_ms = (t_pred_end - t_pred_start) * 1000

y_train_pred = model.predict(X_train)

print(f"Dự đoán trên tập Test (320 mẫu) hoàn tất trong {pred_time_ms:.3f} ms ({pred_time_ms / len(X_test):.4f} ms/mẫu)")

Dự đoán trên tập Test (320 mẫu) hoàn tất trong 1.123 ms (0.0035 ms/mẫu)


## 5. Đánh giá mô hình trên Train set và Test set (MSE, RMSE, MAE, R²)

In [5]:
def evaluate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

train_mse, train_rmse, train_mae, train_r2 = evaluate_metrics(y_train, y_train_pred)
test_mse, test_rmse, test_mae, test_r2 = evaluate_metrics(y_test, y_test_pred)

results_df = pd.DataFrame({
    "Metric": ["MSE", "RMSE", "MAE", "R²"],
    "Train Set": [f"{train_mse:.4f}", f"{train_rmse:.4f}", f"{train_mae:.4f}", f"{train_r2:.4f}"],
    "Test Set": [f"{test_mse:.4f}", f"{test_rmse:.4f}", f"{test_mae:.4f}", f"{test_r2:.4f}"]
})

print("=== KẾT QUẢ ĐÁNH GIÁ BASELINE MODEL ===")
print(results_df.to_string(index=False))
print(f"\nTraining Time: {train_time_ms:.3f} ms")
print(f"Prediction Time (Test): {pred_time_ms:.3f} ms")

=== KẾT QUẢ ĐÁNH GIÁ BASELINE MODEL ===
Metric Train Set Test Set
   MSE    0.4242   0.3900
  RMSE    0.6513   0.6245
   MAE    0.4996   0.5035
    R²    0.3480   0.4032

Training Time: 64.069 ms
Prediction Time (Test): 1.123 ms


## 6. Trích xuất hệ số hồi quy (Weights & Intercept) bàn giao cho Đào (Member 3 - Toán)
Dùng để đối chiếu công thức $y = \mathbf{w}^T \mathbf{x} + b$ trong slide Mathematical Foundation.

In [6]:
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient (w)": model.coef_
}).sort_values(by="Coefficient (w)", ascending=False)

print(f"Hệ số chặn (Intercept b): {model.intercept_:.4f}\n")
print("=== TRỌNG SỐ TỪNG FEATURE (model.coef_) ===")
print(coef_df.to_string(index=False))

Hệ số chặn (Intercept b): 14.3551

=== TRỌNG SỐ TỪNG FEATURE (model.coef_) ===
             Feature  Coefficient (w)
           sulphates         0.841172
             alcohol         0.281890
       fixed acidity         0.023085
      residual sugar         0.006564
 free sulfur dioxide         0.005627
total sulfur dioxide        -0.003644
         citric acid        -0.140821
                  pH        -0.393688
    volatile acidity        -1.001304
           chlorides        -1.806503
             density       -10.351594


## 7. Bàn giao artifacts cho Thư (Member 5 - Experiments & Error Analysis)
Lưu model, bảng metrics, hệ số và kết quả dự đoán test set vào thư mục `models/` và `reports/` để Thư và nhóm tái sử dụng trực tiếp.

In [7]:
# 1. Lưu model đã huấn luyện
model_file = MODELS_DIR / "baseline_linear_regression.joblib"
joblib.dump(model, model_file)

# 2. Lưu bảng metrics
metrics_file = REPORTS_DIR / "baseline_metrics.csv"
results_df.to_csv(metrics_file, index=False, encoding="utf-8-sig")

# 3. Lưu hệ số hồi quy
coef_file = REPORTS_DIR / "baseline_coefficients.csv"
coef_df.to_csv(coef_file, index=False, encoding="utf-8-sig")

# 4. Lưu test predictions và residuals để Thư vẽ Actual vs Predicted & Residual Plot
pred_df = pd.DataFrame({
    "actual_quality": y_test.values,
    "predicted_quality": y_test_pred,
    "residual": y_test.values - y_test_pred
})
pred_file = REPORTS_DIR / "baseline_test_predictions.csv"
pred_df.to_csv(pred_file, index=False, encoding="utf-8-sig")

print("Đã lưu các file bàn giao cho nhóm:")
print(f" - {model_file.relative_to(ROOT)}")
print(f" - {metrics_file.relative_to(ROOT)}")
print(f" - {coef_file.relative_to(ROOT)}")
print(f" - {pred_file.relative_to(ROOT)}")

Đã lưu các file bàn giao cho nhóm:
 - models\baseline_linear_regression.joblib
 - reports\baseline_metrics.csv
 - reports\baseline_coefficients.csv
 - reports\baseline_test_predictions.csv
